In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.0 MB/s eta 0:00:00:00:0100:01


In [2]:
# Dữ liệu 
import pandas as pd
import numpy as np

# Ảnh & PyTorch 
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Tìm kiếm tương đồng
import faiss
from sklearn.metrics.pairwise import cosine_similarity

# Vẽ biểu đồ 
import matplotlib.pyplot as plt
import seaborn as sns

# Tiện ích
import os
from tqdm import tqdm

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 20)      
plt.rcParams['figure.figsize'] = (10, 5)    
sns.set_style('whitegrid')                  


print('Import thư viện thành công!')
print(f'PyTorch version : {torch.__version__}')
# Kiểm tra máy có GPU không. GPU giúp chạy nhanh hơn CPU rất nhiều.
# Laptop bình thường thường sẽ hiện 'cpu' – không sao, vẫn chạy được.
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

Import thư viện thành công!
PyTorch version : 2.11.0+cu128
Device: cuda


DÙNG GG COLAB MƯỢN GPU T4

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

# ĐƯỜNG DẪN ĐÚNG DỰA TRÊN ẢNH GOOGLE DRIVE CỦA BẠN:
DATA_DIR = '/content/drive/MyDrive/DoAnPython/DuLieuPython'

CSV_PATH = os.path.join(DATA_DIR, 'train.csv')
# Vì 'train_images.zip' đang là file nén, bạn cứ khai báo đường dẫn tệp trước:
IMAGE_ZIP_PATH = os.path.join(DATA_DIR, 'train_images.zip') 

PROCESSED = '../data/processed/'
RESULTS = '../results/'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# Kiểm tra lại đường dẫn
for p in [CSV_PATH, IMAGE_ZIP_PATH]:
    status = 'Đã tìm thấy tệp/thư mục' if os.path.exists(p) else 'Không tìm thấy – kiểm tra lại đường dẫn!'
    print(f'{status}  {p}')

Đã tìm thấy tệp/thư mục  /content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv
Đã tìm thấy tệp/thư mục  /content/drive/MyDrive/DoAnPython/DuLieuPython/train_images.zip


TIỀN XỬ LÝ DỮ LIỆU

In [5]:
class ShopeeDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            img_name = self.df.iloc[idx]['image']
            img_path = os.path.join(self.img_dir, img_name)
            image = self.transform(Image.open(img_path).convert("RGB"))
        except Exception:
            image = torch.zeros(3, 224, 224)
            
        text = str(self.df.iloc[idx]['title'])
        text_token = clip.tokenize([text], truncate=True).squeeze(0)
        return image, text_token

In [6]:
df = pd.read_csv(CSV_PATH)

print(f'Train: {df.shape[0]:,} dòng x {df.shape[1]:,} cột')

df.head()

Train: 34,250 dòng x 5 cột


,posting_id,image,image_phash,title,label_group
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045
2,train_2288590299,000a190fdd715a2a36faed16e2c65df7.jpg,b94cb00ed3e50f78,Maling TTS Canned Pork Luncheon Meat 397 gr,2395904891
3,train_2406599165,00117e4fc239b1b641ff08340b429633.jpg,8514fc58eafea283,Daster Batik Lengan pendek - Motif Acak / Camp...,4093212188
4,train_3369186413,00136d1cf4edede0203f32f05f660588.jpg,a6f319f924ad708c,Nescafe \xc3\x89clair Latte 220ml,3648931069


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34250 entries, 0 to 34249
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   posting_id   34250 non-null  object
 1   image        34250 non-null  object
 2   image_phash  34250 non-null  object
 3   title        34250 non-null  object
 4   label_group  34250 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [8]:
candidate_df = pd.read_csv(CSV_PATH)

## 🤖 AI Note – Chọn Model

**DINOv3** (`vit_base_patch16_dinov3.lvd1689m`): Model vision transformer tự giám sát của Meta, được train trên 142M ảnh. Tốt hơn DINOv2 nhờ dữ liệu lớn hơn và kỹ thuật distillation cải tiến.

**paraphrase-multilingual-MiniLM-L12-v2**: Model NLP đa ngôn ngữ (50+ ngôn ngữ bao gồm Tiếng Việt, Indonesia, Thái...). Phù hợp hơn `paraphrase-multilingual-MiniLM-L12-v2` (chỉ tiếng Anh) vì dữ liệu Shopee là đa ngôn ngữ. Kỳ vọng tăng mAP ~3–5%.

In [9]:
!pip install sentence-transformers timm
import torch
from sentence_transformers import SentenceTransformer
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"
# Sửa đổi chuỗi tên định danh DINOv3 chuẩn xác trong thư viện timm
dinov3_model = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=True, num_classes=0).to(device)
dinov3_model.eval()
minilm_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TRÍCH XUẤT VECTO

In [10]:
!unzip -q /content/drive/MyDrive/DoAnPython/DuLieuPython/train_images.zip -d /content/train_images_extracted

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import timm

csv_path = "/content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv"
image_folder = "/content/train_images_extracted/train_images"
output_dir = "/content/drive/MyDrive/DoAnPython/DuLieuPython"

# Đảm bảo thư mục đầu ra tồn tại
os.makedirs(output_dir, exist_ok=True)

candidate_df = pd.read_csv(csv_path)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải mô hình
dinov3_model = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=True, num_classes=0).to(device)
dinov3_model.eval()

# Tự động lấy cấu hình tiền xử lý ảnh chuẩn từ mô hình DINOv3
data_config = timm.data.resolve_model_data_config(dinov3_model)
dinov3_transform = timm.data.create_transform(**data_config, is_training=False)

# 2. Xây dựng Dataset (Bỏ phần xử lý text bên trong để DataLoader chạy nhanh hơn)
class ShopeeImageDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = self.transform(Image.open(img_path).convert("RGB"))
        except Exception:
            # Fallback tạo tensor trống khớp chính xác kích thước yêu cầu của mô hình
            image = torch.zeros(data_config['input_size'])
        return image

dataset = ShopeeImageDataset(candidate_df, image_folder, dinov3_transform)
# Bật pin_memory=True để đẩy dữ liệu lên GPU nhanh hơn
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# 3. Trích xuất đặc trưng HÌNH ẢNH (DINOv3)
image_features_list = []

print("--- Đang trích xuất đặc trưng Hình ảnh (DINOv3) ---")
for images in tqdm(dataloader):
    images = images.to(device, non_blocking=True)
    with torch.no_grad():
        img_features = dinov3_model(images)
    
    # Chuẩn hóa L2 ngay trên GPU
    img_features /= img_features.norm(dim=-1, keepdim=True)
    
    # Chuyển NGAY sang NumPy và ép kiểu float32 để giải phóng bộ nhớ PyTorch
    image_features_list.append(img_features.cpu().numpy().astype(np.float32))

# Gộp các mảng NumPy bằng cách concatenate trực tiếp (tiết kiệm RAM gấp đôi torch.cat)
image_features = np.concatenate(image_features_list, axis=0)
np.save(os.path.join(output_dir, "dinov3_image_features.npy"), image_features)

# Xóa các biến không còn dùng tới để trả lại RAM/VRAM ngay lập tức
del image_features_list, image_features, dinov3_model
torch.cuda.empty_cache()
gc.collect()

# 4. Trích xuất đặc trưng VĂN BẢN (MiniLM)
print("\n--- Đang trích xuất đặc trưng Văn bản (MiniLM) ---")
minilm_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

# Lấy trực tiếp từ DataFrame, xử lý giá trị khuyết (NaN) nếu có bằng fillna
titles_to_encode = candidate_df["title"].fillna("").astype(str).tolist()

text_features = minilm_model.encode(
    titles_to_encode, 
    batch_size=128, # Tăng batch_size cho văn bản để chạy nhanh hơn
    show_progress_bar=True, 
    convert_to_numpy=True
).astype(np.float32)

np.save(os.path.join(output_dir, "minilm_text_features.npy"), text_features)

print("\nHoàn thành! Toàn bộ đặc trưng hình ảnh và văn bản đã được lưu an toàn.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 268/268 [08:50<00:00,  1.98s/it]


Batches:   0%|          | 0/268 [00:00<?, ?it/s]

: 

## 🤖 AI Note – Chia Val/Test Split

Chia dữ liệu thành 3 phần để đánh giá mô hình đúng chuẩn:
- **Train (80%)**: Dùng để trích xuất features
- **Val (10%)**: Dùng để grid search tham số tối ưu (alpha, pHash threshold)
- **Test (10%)**: Dùng để báo cáo metric cuối cùng – **không được dùng để chọn tham số**

Lý do: Nếu dùng toàn bộ data để cả chọn tham số lẫn đánh giá → metric bị overfit, không trung thực.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

csv_path = "/content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv"
output_dir = "/content/drive/MyDrive/DoAnPython/DuLieuPython"

candidate_df = pd.read_csv(csv_path)

# ── Chia val/test split theo label_group (đảm bảo không data leak) ──────────
# Lấy danh sách label_group duy nhất, shuffle ngẫu nhiên
np.random.seed(42)
unique_labels = candidate_df['label_group'].unique()
np.random.shuffle(unique_labels)

n = len(unique_labels)
n_val  = int(n * 0.10)   # 10% label groups cho val
n_test = int(n * 0.10)   # 10% label groups cho test

val_labels  = set(unique_labels[:n_val])
test_labels = set(unique_labels[n_val:n_val + n_test])

val_mask  = candidate_df['label_group'].isin(val_labels)
test_mask = candidate_df['label_group'].isin(test_labels)
train_mask = ~val_mask & ~test_mask

val_df   = candidate_df[val_mask].reset_index(drop=True)
test_df  = candidate_df[test_mask].reset_index(drop=True)
train_df = candidate_df[train_mask].reset_index(drop=True)

print(f"Train: {len(train_df):,} samples | Val: {len(val_df):,} samples | Test: {len(test_df):,} samples")
print(f"Val labels: {len(val_labels)} | Test labels: {len(test_labels)}")

# Lưu index để tra cứu nhanh sau
val_indices  = candidate_df[val_mask].index.values
test_indices = candidate_df[test_mask].index.values


## 🤖 AI Note – Grid Search Alpha + pHash Threshold

Thay vì dùng `alpha = 0.7` cố định:
- **Alpha**: Trọng số kết hợp image/text similarity. `sim = alpha * img_sim + (1-alpha) * txt_sim`
- **pHash threshold**: Ngưỡng Hamming distance để coi 2 ảnh là "giống nhau". Threshold quá thấp (=2) bỏ sót nhiều ảnh tương đồng. Threshold 5–10 capture được nhiều hơn.

Grid search tìm bộ tham số tốt nhất trên **val set** → tránh overfit.

In [ ]:
import gc

# ── Load features đã trích xuất ─────────────────────────────────────────
dinov3_image_features = np.load(os.path.join(output_dir, 'dinov3_image_features.npy')).astype('float32')
minilm_text_features  = np.load(os.path.join(output_dir, 'minilm_text_features.npy')).astype('float32')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
image_tensor = torch.tensor(dinov3_image_features).to(device)
text_tensor  = torch.tensor(minilm_text_features).to(device)

# Chuẩn hóa L2
image_norm = image_tensor / image_tensor.norm(dim=-1, keepdim=True)
text_norm  = text_tensor  / text_tensor.norm(dim=-1,  keepdim=True)

# pHash toàn dataset
phash_strings = candidate_df['image_phash'].values
phash_ints = np.array([int(h, 16) for h in phash_strings], dtype=np.uint64)

# ── Hàm Hamming distance ─────────────────────────────────────────────────
def hamming_batch(batch_ph, all_ph):
    x = batch_ph[:, None] ^ all_ph[None, :]
    x = (x & 0x5555555555555555) + ((x >> 1) & 0x5555555555555555)
    x = (x & 0x3333333333333333) + ((x >> 2) & 0x3333333333333333)
    x = (x & 0x0F0F0F0F0F0F0F0F) + ((x >> 4) & 0x0F0F0F0F0F0F0F0F)
    x = (x & 0x00FF00FF00FF00FF) + ((x >> 8) & 0x00FF00FF00FF00FF)
    x = (x & 0x0000FFFF0000FFFF) + ((x >> 16) & 0x0000FFFF0000FFFF)
    return ((x & 0x00000000FFFFFFFF) + (x >> 32)).astype(np.uint8)

# ── Hàm tính mAP@5 trên subset ───────────────────────────────────────────
def map5_on_subset(subset_idx, alpha, phash_thr, top_k=50, batch=512):
    labels = candidate_df['label_group'].values
    grp = {}
    for i, lb in enumerate(labels):
        grp.setdefault(lb, []).append(i)
    scores = []
    for b0 in range(0, len(subset_idx), batch):
        bidx = subset_idx[b0: b0+batch]
        isim = torch.matmul(image_norm[bidx], image_norm.T).cpu().numpy().astype('float32')
        tsim = torch.matmul(text_norm[bidx],  text_norm.T).cpu().numpy().astype('float32')
        sim  = alpha * isim + (1 - alpha) * tsim
        hd   = hamming_batch(phash_ints[bidx], phash_ints)
        sim[hd <= phash_thr] += 0.5
        sim[hd == 0]         += 0.5
        del isim, tsim, hd
        for li, gi in enumerate(bidx):
            row = sim[li].copy()
            row[gi] = -9999
            top = np.argsort(-row)[:top_k]
            gt  = np.array(grp[labels[gi]])
            gt  = gt[gt != gi]
            if len(gt) == 0:          # Fix 1: skip items không có ground truth
                continue
            rel = np.isin(top[:5], gt)
            pos = np.where(rel)[0] + 1
            ap  = np.sum(np.arange(1, len(pos)+1) / pos) / min(5, len(gt)) if len(pos) else 0.0
            scores.append(ap)
        gc.collect()
    return float(np.mean(scores)) if scores else 0.0

# ── BASELINE với tham số gốc (alpha=0.7, threshold=2) ────────────────────
# Đây là tham số đã cho mAP=0.7388 trong notebook gốc
DEFAULT_ALPHA = 0.7
DEFAULT_THRESH = 2

print('Tính baseline mAP trên val set với tham số gốc (alpha=0.7, threshold=2)...')
try:
    baseline_map = map5_on_subset(val_indices, DEFAULT_ALPHA, DEFAULT_THRESH)
    print(f'  Baseline val mAP@5 = {baseline_map:.4f}')
except NameError:
    baseline_map = 0.0
    print('  (Không có val_indices – chạy cell split trước)')

# ── GRID SEARCH trên VAL SET ─────────────────────────────────────────────
# Chỉ search quanh vùng tốt, tránh các tham số cực đoan
alpha_grid     = [0.6, 0.65, 0.7, 0.75, 0.8]
threshold_grid = [2, 3, 5]   # threshold quá cao (8,10) thường tạo false positives

best_alpha     = DEFAULT_ALPHA
best_threshold = DEFAULT_THRESH
best_map_val   = baseline_map
grid_results   = [{'alpha': DEFAULT_ALPHA, 'threshold': DEFAULT_THRESH,
                   'map5_val': round(baseline_map, 4), 'note': 'baseline'}]

print(f'\nGrid search alpha × pHash threshold trên VAL set...')
print(f"{'Alpha':>6} | {'Thresh':>6} | {'mAP@5':>7} | {'vs baseline':>12}")
print('-' * 42)

try:
    for alpha in alpha_grid:
        for thr in threshold_grid:
            if alpha == DEFAULT_ALPHA and thr == DEFAULT_THRESH:
                continue  # đã tính rồi
            mv = map5_on_subset(val_indices, alpha, thr)
            delta = mv - baseline_map
            grid_results.append({'alpha': alpha, 'threshold': thr,
                                  'map5_val': round(mv, 4), 'note': ''})
            sign = '+' if delta >= 0 else ''
            print(f'{alpha:>6.2f} | {thr:>6d} | {mv:>7.4f} | {sign}{delta:>+.4f}')
            if mv > best_map_val:
                best_map_val   = mv
                best_alpha     = alpha
                best_threshold = thr

    # ── SAFETY CHECK: chỉ dùng kết quả grid search nếu tốt hơn baseline ────
    IMPROVE_THRESHOLD = 0.002  # phải tốt hơn ít nhất 0.2% mới áp dụng
    if best_map_val - baseline_map >= IMPROVE_THRESHOLD:
        print(f'\n✅ Grid search tốt hơn baseline: alpha={best_alpha}, threshold={best_threshold}')
        print(f'   val mAP: {baseline_map:.4f} → {best_map_val:.4f} (+{best_map_val-baseline_map:.4f})')
    else:
        print(f'\n⚠️  Grid search không cải thiện đáng kể (delta < {IMPROVE_THRESHOLD})')
        print(f'   Giữ nguyên tham số gốc: alpha={DEFAULT_ALPHA}, threshold={DEFAULT_THRESH}')
        best_alpha     = DEFAULT_ALPHA
        best_threshold = DEFAULT_THRESH
        best_map_val   = baseline_map

except NameError:
    print('(Không có val_indices – dùng tham số gốc)')
    best_alpha     = DEFAULT_ALPHA
    best_threshold = DEFAULT_THRESH

print(f'\n→ Tham số sẽ dùng: alpha={best_alpha}, threshold={best_threshold}')

# Lưu grid results
grid_df = pd.DataFrame(grid_results)
grid_df.to_csv(os.path.join(output_dir, 'grid_search_results_Hung.csv'), index=False)
print('Đã lưu grid_search_results_Hung.csv')


## TÍNH TOÁN MA TRẬN TƯƠNG ĐỒNG (với tham số tối ưu từ Grid Search)

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

csv_path   = '/content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv'
output_dir = '/content/drive/MyDrive/DoAnPython/DuLieuPython'

candidate_df          = pd.read_csv(csv_path)
dinov3_image_features = np.load(os.path.join(output_dir, 'dinov3_image_features.npy')).astype('float32')
minilm_text_features  = np.load(os.path.join(output_dir, 'minilm_text_features.npy')).astype('float32')

device      = 'cuda' if torch.cuda.is_available() else 'cpu'
image_tensor = torch.tensor(dinov3_image_features).to(device)
text_tensor  = torch.tensor(minilm_text_features).to(device)
image_norm   = image_tensor / image_tensor.norm(dim=-1, keepdim=True)
text_norm    = text_tensor  / text_tensor.norm(dim=-1,  keepdim=True)

phash_strings = candidate_df['image_phash'].values
phash_ints    = np.array([int(h, 16) for h in phash_strings], dtype=np.uint64)

# ── Tham số: ưu tiên dùng kết quả từ grid search (nếu đã chạy) ───────────
# Nếu chưa chạy cell grid search, dùng tham số gốc đã validated
try:
    ALPHA        = best_alpha       # từ grid search cell
    PHASH_THRESH = best_threshold
    print(f'Dùng tham số từ grid search: alpha={ALPHA}, threshold={PHASH_THRESH}')
except NameError:
    ALPHA        = 0.7              # tham số gốc (cho mAP=0.7388)
    PHASH_THRESH = 2                # KHÔNG tự ý tăng threshold
    print(f'Dùng tham số gốc: alpha={ALPHA}, threshold={PHASH_THRESH}')

TOP_K      = 50
BATCH_SIZE = 1024
num_samples = len(candidate_df)

print(f'Tính similarity cho {num_samples:,} samples...')

all_sorted_indices = []
torch.cuda.empty_cache(); gc.collect()

for start_idx in tqdm(range(0, num_samples, BATCH_SIZE)):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)

    img_sim = torch.matmul(image_norm[start_idx:end_idx], image_norm.T)
    txt_sim = torch.matmul(text_norm[start_idx:end_idx],  text_norm.T)
    sim = (ALPHA * img_sim + (1 - ALPHA) * txt_sim).cpu().numpy().astype('float32')
    del img_sim, txt_sim
    torch.cuda.empty_cache()

    # Hamming distance (bit-twiddling)
    bp = phash_ints[start_idx:end_idx]
    x  = bp[:, None] ^ phash_ints[None, :]
    x  = (x & 0x5555555555555555) + ((x >> 1) & 0x5555555555555555)
    x  = (x & 0x3333333333333333) + ((x >> 2) & 0x3333333333333333)
    x  = (x & 0x0F0F0F0F0F0F0F0F) + ((x >> 4) & 0x0F0F0F0F0F0F0F0F)
    x  = (x & 0x00FF00FF00FF00FF) + ((x >> 8) & 0x00FF00FF00FF00FF)
    x  = (x & 0x0000FFFF0000FFFF) + ((x >> 16) & 0x0000FFFF0000FFFF)
    hd = ((x & 0x00000000FFFFFFFF) + (x >> 32)).astype(np.uint8)
    del x

    # pHash boost
    sim[hd <= PHASH_THRESH] += 0.5
    sim[hd == 0]            += 0.5
    del hd

    sorted_idx = np.argsort(-sim, axis=-1)[:, :TOP_K].astype(np.int32)
    all_sorted_indices.append(sorted_idx)
    del sim; gc.collect()

all_sorted_indices = np.vstack(all_sorted_indices)
np.save(os.path.join(output_dir, 'all_sorted_indices.npy'), all_sorted_indices)
print(f'Lưu all_sorted_indices.npy: shape={all_sorted_indices.shape}')


## TÍNH TOÁN METRIC

🤖 **AI Note**: Metric được tính **tách biệt** trên val set và test set:
- **Val set**: Đã dùng để chọn tham số → không dùng làm kết quả báo cáo
- **Test set**: Chưa bao giờ được nhìn thấy → metric trung thực để báo cáo

**Fix quan trọng**: `gt_len = 0` (items không có cặp matching) sẽ bị `continue` – không tính vào mAP vì chúng không có ground truth để đánh giá.

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

csv_path   = '/content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv'
output_dir = '/content/drive/MyDrive/DoAnPython/DuLieuPython'

candidate_df       = pd.read_csv(csv_path)
all_sorted_indices = np.load(os.path.join(output_dir, 'all_sorted_indices.npy'))
labels             = candidate_df['label_group'].values

# Build group map
grp = {}
for i, lb in enumerate(labels):
    grp.setdefault(lb, []).append(i)

def evaluate(subset_indices, split_name='Full'):
    """Tính P@K, R@K, mAP@5. Fix: gt_len=0 → skip (không tính vào mAP)."""
    ap5, p1, r1, p3, r3, p5, r5, p10, r10 = [], [], [], [], [], [], [], [], []
    skipped = 0

    for gi in tqdm(subset_indices, desc=f'Eval {split_name}', leave=False):
        ret  = all_sorted_indices[gi]
        qlb  = labels[gi]
        gt   = np.array(grp[qlb])

        ret  = ret[ret != gi]
        gt   = gt[gt != gi]
        gt_n = len(gt)

        # ✅ Fix 1: bỏ qua items đơn lẻ không có cặp ground truth
        # (Notebook gốc append 0.0 cho những items này → kéo mAP xuống giả tạo)
        if gt_n == 0:
            skipped += 1
            continue

        rel = np.isin(ret, gt)

        h1 = rel[:1].sum();  p1.append(h1/1);   r1.append(h1/gt_n)
        h3 = rel[:3].sum();  p3.append(h3/3);   r3.append(h3/gt_n)
        h5 = rel[:5].sum();  p5.append(h5/5);   r5.append(h5/gt_n)
        h10= rel[:10].sum(); p10.append(h10/10); r10.append(h10/gt_n)

        pos = np.where(rel[:5])[0] + 1
        ap  = np.sum(np.arange(1,len(pos)+1) / pos) / min(5, gt_n) if len(pos) else 0.0
        ap5.append(ap)

    df = pd.DataFrame({
        'K':         [1, 3, 5, 10],
        'Precision': np.round([np.mean(p1),np.mean(p3),np.mean(p5),np.mean(p10)], 4).tolist(),
        'Recall':    np.round([np.mean(r1),np.mean(r3),np.mean(r5),np.mean(r10)], 4).tolist(),
    })
    map5 = round(float(np.mean(ap5)), 4) if ap5 else 0.0
    return df, map5, skipped

# ── Tính trên FULL DATASET (so sánh công bằng với notebook gốc) ──────────
all_idx = np.arange(len(candidate_df))
full_df, full_map5, full_skip = evaluate(all_idx, 'Full')
print('\n📊 FULL DATASET METRICS:')
print(full_df.to_string(index=False))
print(f'mAP@5 = {full_map5}  (bỏ qua {full_skip} items không có ground truth)')
print('(notebook gốc: mAP@5 = 0.7388 — bao gồm cả items không có GT append 0)')

# ── Tính trên VAL SET ────────────────────────────────────────────────────
try:
    val_df, val_map5, val_skip = evaluate(val_indices, 'Val')
    print(f'\n📊 VAL SET METRICS: mAP@5 = {val_map5} (skip {val_skip})')
    print(val_df.to_string(index=False))
except NameError:
    print('\n(Không có val_indices – chạy cell split trước nếu cần)')

# ── Tính trên TEST SET ───────────────────────────────────────────────────
try:
    test_df, test_map5, test_skip = evaluate(test_indices, 'Test')
    print(f'\n📊 TEST SET METRICS: mAP@5 = {test_map5} (skip {test_skip})')
    print(test_df.to_string(index=False))
except NameError:
    print('\n(Không có test_indices – chạy cell split trước nếu cần)')

# ── Lưu CSV ──────────────────────────────────────────────────────────────
full_df['mAP@5'] = full_map5
full_df['model'] = 'DINOv3 + MultilingualMiniLM'
full_df['split'] = 'full'
csv_out = os.path.join(output_dir, 'metrics_Hung_DINOv3_MultilingualMiniLM.csv')
full_df.to_csv(csv_out, index=False)
print(f'\n✅ Lưu metric: {csv_out}')
